# Office365 工具包

>[Microsoft 365](https://www.office.com/) 是由 `Microsoft` 拥有的生产力软件、协作和基于云的服务的产品系列。
>
>注意：`Office 365` 已更名为 `Microsoft 365`。

本笔记本将引导您完成将 LangChain 连接到 `Office365` 电子邮件和日历。

要使用此工具包，您需要设置凭据，具体说明请参阅 [Microsoft Graph 身份验证和授权概述](https://learn.microsoft.com/en-us/graph/auth/)。在收到 CLIENT_ID 和 CLIENT_SECRET 后，您可以将它们作为环境变量输入到下方。

您也可以使用[此处的身份验证说明](https://o365.github.io/python-o365/latest/getting_started.html#oauth-setup-pre-requisite)。

In [1]:
%pip install --upgrade --quiet  O365
%pip install --upgrade --quiet  beautifulsoup4  # This is optional but is useful for parsing HTML messages
%pip install -qU langchain-community

## 分配环境变量

该工具包将读取 `CLIENT_ID` 和 `CLIENT_SECRET` 环境变量以对用户进行身份验证，因此您需要在此处设置它们。您还需要设置 `OPENAI_API_KEY` 以便稍后使用该代理。

In [ ]:
# Set environmental variables here

## 创建工具包并获取工具

首先，您需要创建工具包，以便稍后访问其工具。

In [3]:
from langchain_community.agent_toolkits import O365Toolkit

toolkit = O365Toolkit()
tools = toolkit.get_tools()
tools

[O365SearchEvents(name='events_search', description=" Use this tool to search for the user's calendar events. The input must be the start and end datetimes for the search query. The output is a JSON list of all the events in the user's calendar between the start and end times. You can assume that the user can  not schedule any meeting over existing meetings, and that the user is busy during meetings. Any times without events are free for the user. ", args_schema=<class 'langchain_community.tools.office365.events_search.SearchEventsInput'>, return_direct=False, verbose=False, callbacks=None, callback_manager=None, handle_tool_error=False, account=Account Client Id: f32a022c-3c4c-4d10-a9d8-f6a9a9055302),
 O365CreateDraftMessage(name='create_email_draft', description='Use this tool to create a draft email with the provided message fields.', args_schema=<class 'langchain_community.tools.office365.create_draft_message.CreateDraftMessageSchema'>, return_direct=False, verbose=False, callbacks

## 在 Agent 中使用

You can use the `Agent` class to create an agent that can use tools.

您可以使用 `Agent` 类来创建一个可以使用工具的 Agent。

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.llms import OpenAI
from langchain_community.tools import DuckDuckGoSearchRun

# Define the tools the agent will use
tools = [DuckDuckGoSearchRun()]

# Define the LLM
llm = OpenAI(temperature=0)

# Create the agent
# The agent_type is not needed when using create_react_agent
agent = create_react_agent(llm, tools)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the capital of France?"})
```

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.llms import OpenAI
from langchain_community.tools import DuckDuckGoSearchRun

# Define the tools the agent will use
tools = [DuckDuckGoSearchRun()]

# Define the LLM
llm = OpenAI(temperature=0)

# Create the agent
# The agent_type is not needed when using create_react_agent
agent = create_react_agent(llm, tools)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the capital of France?"})
```

This will output:

这将输出：

```text
> Entering new AgentExecutor chain...
What is the capital of France?
I need to find out the capital of France. I can use a search engine for this.
Final Answer: The capital of France is Paris.

> Finished chain.
```

```text
> Entering new AgentExecutor chain...
What is the capital of France?
I need to find out the capital of France. I can use a search engine for this.
Final Answer: The capital of France is Paris.

> Finished chain.
```

### Agent Types

LangChain offers several types of agents, each suited for different use cases.

LangChain 提供了几种类型的 Agent，每种都适用于不同的用例。

*   **`zero-shot-react-description`**: This agent uses the ReAct framework and a zero-shot approach. It relies on the tool's description to understand how to use it. This is a good general-purpose agent.
    *   **`zero-shot-react-description`**: 此 Agent 使用 ReAct 框架和零样本方法。它依赖于工具的描述来理解如何使用该工具。这是一个很好的通用 Agent。
*   **`react-docstore`**: This agent also uses the ReAct framework but is specifically designed for use with the `Docstore` tool. It's useful for agents that need to retrieve information from a document store.
    *   **`react-docstore`**: 此 Agent 也使用 ReAct 框架，但专门为与 `Docstore` 工具一起使用而设计。它对于需要从文档存储中检索信息的 Agent 非常有用。
*   **`self-ask-with-search`**: This agent uses a self-asking approach, where it asks itself follow-up questions to break down a complex query. It's often used with search tools.
    *   **`self-ask-with-search`**: 此 Agent 使用一种自我提问的方法，它会问自己后续问题来分解复杂查询。它通常与搜索工具一起使用。
*   **`conversational-react-description`**: This agent is designed for conversational use cases. It can maintain context and engage in multi-turn conversations while using tools.
    *   **`conversational-react-description`**: 此 Agent 专为对话用例而设计。它可以在使用工具的同时维护上下文并进行多轮对话。
*   **`structured-chat-zero-shot-react-description`**: This agent is similar to `zero-shot-react-description` but is optimized for chat models and structured output.
    *   **`structured-chat-zero-shot-react-description`**: 此 Agent 类似于 `zero-shot-react-description`，但针对聊天模型和结构化输出了进行了优化。
*   **`openai-functions`**: This agent leverages OpenAI's function calling capabilities to determine which tool to use. It's particularly effective when working with OpenAI models that support function calling.
    *   **`openai-functions`**: 此 Agent 利用 OpenAI 的函数调用功能来确定使用哪个工具。当与支持函数调用的 OpenAI 模型配合使用时，它特别有效。

You can create an agent using a specific agent type like this:

您可以通过以下方式使用特定的 Agent 类型来创建 Agent：

```python
from langchain.agents import AgentExecutor, create_agent
from langchain_community.llms import OpenAI
from langchain_community.tools import DuckDuckGoSearchRun

# Define the tools the agent will use
tools = [DuckDuckGoSearchRun()]

# Define the LLM
llm = OpenAI(temperature=0)

# Create the agent using a specific agent type
agent = create_agent(llm, tools, agent_type="zero-shot-react-description")

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the capital of France?"})
```

```python
from langchain.agents import AgentExecutor, create_agent
from langchain_community.llms import OpenAI
from langchain_community.tools import DuckDuckGoSearchRun

# Define the tools the agent will use
tools = [DuckDuckGoSearchRun()]

# Define the LLM
llm = OpenAI(temperature=0)

# Create the agent using a specific agent type
agent = create_agent(llm, tools, agent_type="zero-shot-react-description")

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the capital of France?"})
```

### Customizing Agents

You can customize agents by providing a custom prompt, a different LLM, or a different set of tools.

您可以通过提供自定义提示、不同的 LLM 或不同的工具集来定制 Agent。

#### Custom Prompt

Here's an example of how to create an agent with a custom prompt:

以下是如何使用自定义提示创建 Agent 的示例：

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import OpenAI
from langchain_community.tools import DuckDuckGoSearchRun

# Define the tools the agent will use
tools = [DuckDuckGoSearchRun()]

# Define the LLM
llm = OpenAI(temperature=0)

# Define a custom prompt template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the question

Begin!

Question: {input}
{agent_scratchpad}
""")

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the capital of France?"})
```

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import OpenAI
from langchain_community.tools import DuckDuckGoSearchRun

# Define the tools the agent will use
tools = [DuckDuckGoSearchRun()]

# Define the LLM
llm = OpenAI(temperature=0)

# Define a custom prompt template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the question

Begin!

Question: {input}
{agent_scratchpad}
""")

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the capital of France?"})
```

#### Custom LLM

You can also use a different LLM, such as one from Hugging Face:

您也可以使用不同的 LLM，例如来自 Hugging Face 的 LLM：

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.llms import HuggingFaceHub
from langchain_community.tools import DuckDuckGoSearchRun

# Define the tools the agent will use
tools = [DuckDuckGoSearchRun()]

# Define the LLM
llm = HuggingFaceHub(repo_id="google/flan-t5-large", model_kwargs={"temperature": 0})

# Create the agent
agent = create_react_agent(llm, tools)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the capital of France?"})
```

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.llms import HuggingFaceHub
from langchain_community.tools import DuckDuckGoSearchRun

# Define the tools the agent will use
tools = [DuckDuckGoSearchRun()]

# Define the LLM
llm = HuggingFaceHub(repo_id="google/flan-t5-large", model_kwargs={"temperature": 0})

# Create the agent
agent = create_react_agent(llm, tools)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the capital of France?"})
```

#### Custom Tools

You can also create your own custom tools and use them with an agent. Here's an example of a custom tool:

您还可以创建自己的自定义工具并将其与 Agent 一起使用。以下是一个自定义工具的示例：

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.tools import tool
from langchain_community.llms import OpenAI

@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

# Define the tools the agent will use
tools = [get_word_length]

# Define the LLM
llm = OpenAI(temperature=0)

# Create the agent
agent = create_react_agent(llm, tools)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the length of the word 'apple'?"})
```

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.tools import tool
from langchain_community.llms import OpenAI

@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

# Define the tools the agent will use
tools = [get_word_length]

# Define the LLM
llm = OpenAI(temperature=0)

# Create the agent
agent = create_react_agent(llm, tools)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
agent_executor.invoke({"input": "What is the length of the word 'apple'?"})

In [4]:
from langchain.agents import AgentType, initialize_agent
from langchain_openai import OpenAI

In [5]:
llm = OpenAI(temperature=0)
agent = initialize_agent(
    tools=toolkit.get_tools(),
    llm=llm,
    verbose=False,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
)

In [6]:
agent.run(
    "Create an email draft for me to edit of a letter from the perspective of a sentient parrot"
    " who is looking to collaborate on some research with her"
    " estranged friend, a cat. Under no circumstances may you send the message, however."
)

'The draft email was created correctly.'

In [12]:
agent.run(
    "Could you search in my drafts folder and let me know if any of them are about collaboration?"
)

"I found one draft in your drafts folder about collaboration. It was sent on 2023-06-16T18:22:17+0000 and the subject was 'Collaboration Request'."

In [8]:
agent.run(
    "Can you schedule a 30 minute meeting with a sentient parrot to discuss research collaborations on October 3, 2023 at 2 pm Easter Time?"
)

/home/vscode/langchain-py-env/lib/python3.11/site-packages/O365/utils/windows_tz.py:639: PytzUsageWarning: The zone attribute is specific to pytz's interface; please migrate to a new time zone provider. For more details on how to do so, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  iana_tz.zone if isinstance(iana_tz, tzinfo) else iana_tz)
/home/vscode/langchain-py-env/lib/python3.11/site-packages/O365/utils/utils.py:463: PytzUsageWarning: The zone attribute is specific to pytz's interface; please migrate to a new time zone provider. For more details on how to do so, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  timezone = date_time.tzinfo.zone if date_time.tzinfo is not None else None


'I have scheduled a meeting with a sentient parrot to discuss research collaborations on October 3, 2023 at 2 pm Easter Time. Please let me know if you need to make any changes.'

In [9]:
agent.run(
    "Can you tell me if I have any events on October 3, 2023 in Eastern Time, and if so, tell me if any of them are with a sentient parrot?"
)

"Yes, you have an event on October 3, 2023 with a sentient parrot. The event is titled 'Meeting with sentient parrot' and is scheduled from 6:00 PM to 6:30 PM."